## Generating RAG Answers

In [2]:
# Now we evaluate the full RAG pipeline. For each generated question, we run RAG and save the answer produced by the LLM. 
# Later, we'll compare this answer with the original FAQ answer.

# This is the A->Q->A' setup:

# A = original answer in the FAQ
# Q = generated question from this answer
# A' = answer produced by our RAG system
# If A' is close to A, the RAG system is doing a good job.

# This is still offline evaluation. We can compare A and A' because our questions came from FAQ records. 
# For each question, we know which original answer it came from.

# Loading the data
# Create a new notebook for RAG evaluation.

# Load the ground truth questions:

import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
# Load the FAQ documents and the search index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# Create a lookup table for the original FAQ documents:

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc
    
# We'll use this lookup table to find the original answer for each ground truth question.

In [5]:
# Running RAG
# Import the usual things first:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

